# Memory: Compacting Long Conversations

An LLM's context window is finite, but a chat session can run far longer than that window allows. **Compaction** is the fix: once a conversation gets long, summarize the older turns into a dense paragraph, keep only the most recent messages verbatim, and persist that compacted state to disk.

This notebook simulates two sessions:
1. **Session 1** — load a long raw history, compact it once it crosses a token threshold, and save the result.
2. **Session 2** — a fresh process restores *only* the compacted state (summary + recent messages) and keeps the conversation going, proving the summary preserved the facts that matter.

---
## Step 1: Install Dependencies

In [ ]:
!pip install -q openai python-dotenv

---
## Step 2: Set Up Environment Variables & Client

In [ ]:
import os
from getpass import getpass
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

if "OPENAI_API_KEY" not in os.environ or not os.environ["OPENAI_API_KEY"]:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API Key: ")

client = OpenAI()
MODEL = "gpt-4o-mini"

---
## Step 3: Seed a Long Conversation History
In a real app this file would already exist on disk after many turns of chatting. Here we write it out ourselves so the notebook is self-contained: a design discussion for a fictional service, `HEL-292`, packed with concrete facts (framework, database, auth scheme, deployment, rate limits) that a good summary must preserve.

In [ ]:
%%writefile chat_history.json
[
  {"role": "user", "content": "Let's build a service called HEL-292 using FastAPI."},
  {"role": "assistant", "content": "Got it. HEL-292 will be built with FastAPI for structured, async-friendly APIs."},
  {"role": "user", "content": "We'll also add a PostgreSQL database with the pgvector extension, Pydantic for settings, and background tasks for model inference."},
  {"role": "assistant", "content": "Updated architecture: FastAPI, PostgreSQL + pgvector, Pydantic config, and async background workers for inference."},
  {"role": "user", "content": "For auth, let's go with JWT access tokens plus refresh tokens, issued via an OAuth2 password flow."},
  {"role": "assistant", "content": "Noted: JWT access + refresh tokens, OAuth2 password flow for login."},
  {"role": "user", "content": "Deployment should be Docker containers on AWS ECS, behind an Application Load Balancer, with secrets in AWS Secrets Manager."},
  {"role": "assistant", "content": "Deployment plan: Docker on ECS, behind an ALB, secrets pulled from AWS Secrets Manager."},
  {"role": "user", "content": "For testing, use pytest with httpx's AsyncClient, and run the suite in GitHub Actions on every PR."},
  {"role": "assistant", "content": "Testing: pytest + httpx AsyncClient, CI via GitHub Actions on every pull request."},
  {"role": "user", "content": "We need rate limiting too — 100 requests per minute per API key, using a Redis-backed sliding window."},
  {"role": "assistant", "content": "Added: rate limiting at 100 req/min per API key, Redis-backed sliding window."},
  {"role": "user", "content": "For observability, wire up Prometheus metrics and a Grafana dashboard for latency and error rate."},
  {"role": "assistant", "content": "Observability: Prometheus metrics exposed, Grafana dashboard tracking latency and error rate."},
  {"role": "user", "content": "Let's call the primary metrics endpoint /internal/metrics and keep it off the public router."},
  {"role": "assistant", "content": "Done — /internal/metrics will be mounted separately from the public API router."}
]

---
## Step 4: Configuration & Helpers
`COMPACTION_TOKEN_THRESHOLD` is set artificially low so the demo actually triggers compaction quickly and stays cheap to run. In a real app this would be a real token count checked against the model's context window. `KEEP_RECENT` messages are always kept verbatim — only what's older gets folded into a summary.

In [ ]:
import json

HISTORY_FILE = "chat_history.json"
STORE_FILE = "chat_store.json"
SESSION_KEY = "user_session_123"

COMPACTION_TOKEN_THRESHOLD = 150
KEEP_RECENT = 4  # always keep the last N messages verbatim, uncompacted


def approx_tokens(messages):
    return sum(len(m["content"]) for m in messages) // 4


def ask(messages, **kwargs):
    return client.chat.completions.create(model=MODEL, messages=messages, **kwargs).choices[0].message.content

---
## Step 5: The Compaction Function
Everything except the last `KEEP_RECENT` messages gets summarized into one dense paragraph — instructed to preserve every concrete fact, decision, name, number, and technology choice rather than generalize. Nothing is silently dropped; it's folded into the summary first.

In [ ]:
def compact(history):
    to_summarize, recent = history[:-KEEP_RECENT], history[-KEEP_RECENT:]

    transcript = "\n".join(f"{m['role']}: {m['content']}" for m in to_summarize)
    summary = ask([
        {
            "role": "system",
            "content": (
                "Summarize this conversation into a dense paragraph for use as "
                "long-term memory. Preserve every concrete fact, decision, name, "
                "number, and technology choice — do not generalize or omit specifics."
            ),
        },
        {"role": "user", "content": transcript},
    ])
    return summary, recent

---
## Step 6: Session 1 — Load, Compact, Persist
Load the raw history from disk, check whether it crosses the compaction threshold, and if so, replace the old messages with a summary. The compacted state — summary, recent messages, and some bookkeeping — is written to `chat_store.json`.

In [ ]:
print("=== SESSION 1: load raw history, compact it, persist to disk ===\n")

with open(HISTORY_FILE) as f:
    history = json.load(f)

print(f"Loaded {len(history)} messages from {HISTORY_FILE} (~{approx_tokens(history)} tokens)")

if approx_tokens(history) < COMPACTION_TOKEN_THRESHOLD:
    print("Under the compaction threshold — nothing to compact.")
    summary, recent = None, history
else:
    summary, recent = compact(history)
    print(f"\nCompacted {len(history) - KEEP_RECENT} older messages into a summary:")
    print(f"  \"{summary}\"")
    print(f"\nKept the last {KEEP_RECENT} messages verbatim.")

store = {
    "key": SESSION_KEY,
    "summary": summary,
    "recent_messages": recent,
    "original_message_count": len(history),
    "original_tokens": approx_tokens(history),
    "compacted_tokens": approx_tokens(recent) + (len(summary) // 4 if summary else 0),
}
with open(STORE_FILE, "w") as f:
    json.dump(store, f, indent=2)

print(f"\nSaved compacted state to {STORE_FILE}: "
      f"{store['original_tokens']} tokens -> {store['compacted_tokens']} tokens")

---
## Step 7: Session 2 — Restore & Keep Chatting
Simulate a brand-new process that never saw the raw history — it only reads `chat_store.json`. We rebuild the message list from the summary plus the recent messages, then ask a question whose answer depends on facts buried in the *summarized* part of the conversation.

In [ ]:
print("=" * 60)
print("=== SESSION 2: restore compacted state and keep chatting ===")
print("=" * 60 + "\n")

with open(STORE_FILE) as f:
    restored = json.load(f)

messages = [{"role": "system", "content": "You are a helpful engineering assistant."}]
if restored["summary"]:
    messages.append({"role": "system", "content": f"Earlier conversation summary: {restored['summary']}"})
messages.extend(restored["recent_messages"])

print(f"Restored context: 1 summary block + {len(restored['recent_messages'])} recent messages "
      f"(~{approx_tokens(messages)} tokens) — the original {restored['original_message_count']} "
      f"raw messages are gone, only their summary survives.\n")

test_query = "What database and framework did we select for HEL-292, and what's the rate limit?"
messages.append({"role": "user", "content": test_query})
response = ask(messages)

print(f"User: {test_query}")
print(f"Assistant: {response}")